# BBO Capstone — Initial Data Exploration

**Goal of this notebook:** load the initial samples for all 8 black-box functions
and *get a feel* for each one before choosing a Bayesian-Optimisation strategy
(kernel + acquisition function) per function.

Every function is a **maximisation** problem (some were re-framed from a
minimisation, e.g. maximising the *negative* of side-effects / penalties).
All inputs live in the unit hypercube **[0, 1]^d** (the data confirms this),
so no rescaling is needed before fitting a GP.

For each function you get:
1. A short **description summary** (as comments + markdown) with the key
   structural hints — modality, noise, dimensionality, sparsity.
2. A **data summary** (shape, per-dimension ranges, output range, best point so far).
3. **Scatter plots**: one marginal panel per input dimension vs. the output,
   the best-so-far point highlighted. 2-D functions also get a proper 2-D
   colour-map view; higher-D functions get a pair-plot + parallel-coordinates view.

> Run top-to-bottom. Set `DATA_DIR` in the config cell if your `initial_data`
> folder is somewhere other than next to this notebook.

In [ ]:
# =====================================================================
# Config & imports
# =====================================================================
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm

# --- Where is the data? -----------------------------------------------
# The notebook expects a folder layout like:
#   <DATA_DIR>/function_1/initial_inputs.npy
#   <DATA_DIR>/function_1/initial_outputs.npy
#   ... up to function_8
# By default we look for an "initial_data" folder next to this notebook.
DATA_DIR = Path("initial_data")

# Fallback: also try the current working directory if the default is missing.
if not DATA_DIR.exists():
    for cand in [Path.cwd() / "initial_data", Path.cwd()]:
        if (cand / "function_1").exists():
            DATA_DIR = cand
            break

print("Using DATA_DIR =", DATA_DIR.resolve())
assert (DATA_DIR / "function_1").exists(), (
    f"Could not find function folders under {DATA_DIR.resolve()!s}. "
    "Edit DATA_DIR above to point at your initial_data folder."
)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
N_FUNCS = 8

In [ ]:
# =====================================================================
# Function metadata  --  SUMMARISED DESCRIPTIONS
# =====================================================================
# Each entry captures the key hints from the official description that tell us
# something about the function's SHAPE, and therefore how to optimise it.
# `dim` = input dimensionality, `n` = number of initial points supplied.
# `goal` is always "maximise" (some were re-framed from minimisation).
#
# The "shape hints" are what matter for picking a kernel + acquisition function:
#   - unimodal   -> exploit hard (e.g. plain EI / low-xi), fast to solve
#   - multimodal -> explore more (UCB with larger beta, or EI with higher xi)
#   - noisy      -> use a noise term (learn likelihood / WhiteKernel), prefer
#                   noise-robust acquisition (e.g. noisy-EI); don't trust single reads
#   - sparse     -> most of the space is flat/zero, signal is a needle-in-haystack
#                   -> exploration-heavy, space-filling starts help a lot
#   - high-dim   -> global search is hard; target strong LOCAL maxima, consider
#                   trust-region BO (TuRBO) / good priors on the search space

FUNCTIONS = {
    1: dict(dim=2, n=10, goal="maximise",
            title="Contamination-source detection (2-D radiation field)",
            hints=[
                "MAXIMISE.  Detect likely contamination/radiation sources in a 2-D area.",
                "Only PROXIMITY to a source yields a non-zero reading -> the surface is",
                "  mostly FLAT/ZERO with a few sharp peaks => a SPARSE, needle-in-haystack signal.",
                "Must find BOTH strong AND weak sources -> don't stop at the first peak.",
                "STRATEGY: exploration-heavy. Space-filling / high-beta UCB to hunt for any",
                "  non-zero region first, then refine. Watch: initial outputs here are ~0.",
            ]),
    2: dict(dim=2, n=10, goal="maximise",
            title="Noisy log-likelihood (2-D, many local peaks)",
            hints=[
                "MAXIMISE a log-likelihood score of a mystery ML model (2 inputs).",
                "Output is NOISY, and the surface is MULTIMODAL with many local peaks",
                "  -> easy to get stuck in a local optimum depending on where you start.",
                "STRATEGY: classic explore/exploit trade-off. Fit a GP WITH a noise term",
                "  (WhiteKernel / learned likelihood). Use EI with some exploration (xi>0)",
                "  or UCB; prefer noise-aware acquisition. Don't over-trust a single read.",
            ]),
    3: dict(dim=3, n=15, goal="maximise",
            title="Drug discovery — minimise side-effects (3 compounds)",
            hints=[
                "3 compounds per experiment; output = number of adverse reactions.",
                "TRUE goal is to MINIMISE side-effects, RE-FRAMED as MAXIMISE the negative",
                "  of side-effects (so the stored outputs are already negative -> maximise them).",
                "Costly wet-lab experiments -> sample-efficient BO matters.",
                "STRATEGY: standard EI/UCB on the transformed (negative) output; each point",
                "  is expensive so favour informative picks.",
            ]),
    4: dict(dim=4, n=30, goal="maximise",
            title="Warehouse product placement (4 hyperparameters)",
            hints=[
                "MAXIMISE. Output = difference from an EXPENSIVE biweekly baseline calc.",
                "4 hyperparameters tuning an ML surrogate of a costly, DYNAMIC system.",
                "System is 'full of LOCAL OPTIMA' -> MULTIMODAL, needs robust validation.",
                "STRATEGY: explore before exploiting; UCB / EI-with-exploration. Beware",
                "  over-fitting to a single local optimum; validate promising regions.",
            ]),
    5: dict(dim=4, n=20, goal="maximise",
            title="Chemical-process yield (4-D, unimodal)",
            hints=[
                "MAXIMISE yield of a chemical process (4 inputs).",
                "Function is 'typically UNIMODAL, with a SINGLE PEAK' -> the EASY case.",
                "STRATEGY: exploit! Plain EI (small xi) or low-beta UCB converges fast.",
                "  A smooth (RBF / Matern-5/2) kernel fits well. Note: output range here is",
                "  huge (up to ~1000), so consider log/standardising the target for GP stability.",
            ]),
    6: dict(dim=5, n=20, goal="maximise",
            title="Cake-recipe score (5 ingredients, negative-by-design)",
            hints=[
                "5 ingredient inputs (flour, sugar, eggs, butter, milk...).",
                "Score is a SUM of NEGATIVE penalties (flavour/consistency/calories/waste/cost)",
                "  so the total is NEGATIVE BY DESIGN. Goal: bring score AS CLOSE TO 0 as",
                "  possible == MAXIMISE the (negative) sum.",
                "STRATEGY: standard maximisation on the raw (negative) output. Additive",
                "  structure hints a smooth surface; Matern kernel + EI is a safe default.",
            ]),
    7: dict(dim=6, n=30, goal="maximise",
            title="ML hyperparameter tuning (6-D, accuracy/F1 in [0,1])",
            hints=[
                "MAXIMISE a model performance score (accuracy / F1), so output is ~[0,1].",
                "6 hyperparameters (learning rate, regularisation, #hidden layers, ...).",
                "Because this is a COMMON setup, prior LITERATURE can guide the search space",
                "  (e.g. log-scale learning rate) -> good priors help a lot.",
                "STRATEGY: EI/UCB; consider input warping (log) for scale-sensitive params.",
            ]),
    8: dict(dim=8, n=40, goal="maximise",
            title="High-dimensional black-box (8-D ML tuning)",
            hints=[
                "MAXIMISE an 8-D black box (e.g. 8 ML hyperparameters).",
                "HIGH-DIMENSIONAL & complex -> global optimisation is HARD; the description",
                "  says identifying STRONG LOCAL MAXIMA is the practical strategy.",
                "STRATEGY: trust-region BO (TuRBO) or local-restart BO; UCB with moderate",
                "  exploration. Don't expect to cover the whole space with a handful of points.",
            ]),
}

# quick sanity print
for i, m in FUNCTIONS.items():
    print(f"F{i}: {m['dim']}-D, {m['n']} pts, {m['goal']:8s} | {m['title']}")

In [ ]:
# =====================================================================
# Load all functions into a dict of {index: (X, y)}
# =====================================================================
def load_function(i):
    """Return (X, y) for function i.  X shape (n, dim), y shape (n,)."""
    folder = DATA_DIR / f"function_{i}"
    X = np.load(folder / "initial_inputs.npy")
    y = np.load(folder / "initial_outputs.npy").ravel()
    return X, y

DATA = {i: load_function(i) for i in range(1, N_FUNCS + 1)}

for i, (X, y) in DATA.items():
    assert X.shape[0] == y.shape[0], f"F{i}: mismatched #rows"
    assert X.shape[1] == FUNCTIONS[i]["dim"], f"F{i}: dim mismatch"
print("Loaded all 8 functions OK.")

In [ ]:
# =====================================================================
# High-level overview table for all 8 functions
# =====================================================================
rows = []
for i, (X, y) in DATA.items():
    b = int(np.argmax(y))  # best (we always MAXIMISE)
    rows.append({
        "func": f"F{i}",
        "dim": X.shape[1],
        "n_pts": X.shape[0],
        "x_min": round(float(X.min()), 3),
        "x_max": round(float(X.max()), 3),
        "y_min": round(float(y.min()), 4),
        "y_max (best)": round(float(y.max()), 4),
        "y_mean": round(float(y.mean()), 4),
        "y_std": round(float(y.std()), 4),
        "best_x": np.round(X[b], 3).tolist(),
        "shape hint": FUNCTIONS[i]["title"],
    })
overview = pd.DataFrame(rows).set_index("func")
pd.set_option("display.max_colwidth", 60)
overview

In [ ]:
# =====================================================================
# Plotting helpers
# =====================================================================
def print_summary(i):
    X, y = DATA[i]
    m = FUNCTIONS[i]
    b = int(np.argmax(y))
    print("=" * 70)
    print(f"FUNCTION {i}: {m['title']}")
    print(f"  dim={X.shape[1]}  n_points={X.shape[0]}  goal={m['goal']}")
    print("-" * 70)
    for h in m["hints"]:
        print("  " + h)
    print("-" * 70)
    print(f"  output: min={y.min():.4g}  max={y.max():.4g}  "
          f"mean={y.mean():.4g}  std={y.std():.4g}")
    print(f"  BEST so far (max): y={y[b]:.4g} at x={np.round(X[b],4).tolist()}")
    for d in range(X.shape[1]):
        print(f"    x{d}: range [{X[:,d].min():.3f}, {X[:,d].max():.3f}]")
    print("=" * 70)


def plot_marginals(i, cmap="viridis"):
    """One panel per input dim: x_d vs output, coloured by output.
    The best-so-far point is marked with a red star."""
    X, y = DATA[i]
    m = FUNCTIONS[i]
    d = X.shape[1]
    b = int(np.argmax(y))
    ncols = min(d, 4)
    nrows = int(np.ceil(d / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(3.4 * ncols, 3.1 * nrows),
                             squeeze=False)
    for k in range(nrows * ncols):
        ax = axes[k // ncols][k % ncols]
        if k >= d:
            ax.axis("off")
            continue
        sc = ax.scatter(X[:, k], y, c=y, cmap=cmap, s=55,
                        edgecolor="k", linewidth=0.4)
        ax.scatter(X[b, k], y[b], marker="*", s=320, c="red",
                   edgecolor="k", linewidth=0.6, zorder=5, label="best")
        ax.set_xlabel(f"x{k}")
        ax.set_ylabel("output" if k % ncols == 0 else "")
        if k == 0:
            ax.legend(loc="best", fontsize=8)
    fig.colorbar(sc, ax=axes.ravel().tolist(), shrink=0.8, label="output")
    fig.suptitle(f"F{i} — {m['title']}\n(each input dimension vs. output; "
                 f"red ★ = best point)", fontsize=11)
    plt.show()


def plot_2d_field(i, cmap="viridis"):
    """For 2-D functions: x0 vs x1, coloured/sized by output."""
    X, y = DATA[i]
    m = FUNCTIONS[i]
    b = int(np.argmax(y))
    fig, ax = plt.subplots(figsize=(5.2, 4.3))
    # size scales with rank so near-equal (e.g. sparse ~0) outputs stay visible
    order = np.argsort(np.argsort(y))
    sizes = 40 + 260 * (order / max(order.max(), 1))
    sc = ax.scatter(X[:, 0], X[:, 1], c=y, s=sizes, cmap=cmap,
                    edgecolor="k", linewidth=0.5)
    ax.scatter(X[b, 0], X[b, 1], marker="*", s=380, c="red",
               edgecolor="k", linewidth=0.7, zorder=5, label="best")
    for j in range(len(X)):
        ax.annotate(f"{y[j]:.2g}", (X[j, 0], X[j, 1]),
                    fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel("x0"); ax.set_ylabel("x1")
    ax.set_title(f"F{i} — 2-D input space (colour & size = output)")
    ax.legend(loc="best", fontsize=8)
    fig.colorbar(sc, ax=ax, label="output")
    plt.show()


def plot_pairs(i, cmap="viridis"):
    """For higher-D functions: pairwise scatter matrix of input dims,
    coloured by output. Diagonal shows that dim vs output."""
    X, y = DATA[i]
    d = X.shape[1]
    fig, axes = plt.subplots(d, d, figsize=(1.9 * d, 1.9 * d), squeeze=False)
    for r in range(d):
        for c in range(d):
            ax = axes[r][c]
            if r == c:
                ax.scatter(X[:, c], y, c=y, cmap=cmap, s=18,
                           edgecolor="k", linewidth=0.2)
                ax.set_facecolor("#f5f5f5")
            else:
                ax.scatter(X[:, c], X[:, r], c=y, cmap=cmap, s=18,
                           edgecolor="k", linewidth=0.2)
            if r == d - 1:
                ax.set_xlabel(f"x{c}", fontsize=8)
            if c == 0:
                ax.set_ylabel(f"x{r}" if r != c else "out", fontsize=8)
            ax.tick_params(labelsize=6)
    fig.suptitle(f"F{i} — pairwise input scatter (colour = output; "
                 f"diagonal = dim vs output)", fontsize=11)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


def plot_parallel(i, cmap="viridis"):
    """Parallel-coordinates view: each sample is a line across the input dims,
    coloured by its output. Good for spotting which regions give high output."""
    X, y = DATA[i]
    d = X.shape[1]
    fig, ax = plt.subplots(figsize=(1.3 * d + 3, 4))
    norm = (y - y.min()) / (np.ptp(y) + 1e-12)
    colors = cm.get_cmap(cmap)(norm)
    xs = np.arange(d)
    for j in range(len(X)):
        ax.plot(xs, X[j], color=colors[j], alpha=0.75, lw=1.6)
    b = int(np.argmax(y))
    ax.plot(xs, X[b], color="red", lw=2.6, label="best", zorder=5)
    ax.set_xticks(xs); ax.set_xticklabels([f"x{k}" for k in range(d)])
    ax.set_ylabel("input value (all in [0,1])")
    ax.set_title(f"F{i} — parallel coordinates (line colour = output; red = best)")
    ax.legend(loc="best", fontsize=8)
    sm = cm.ScalarMappable(cmap=cmap,
                           norm=plt.Normalize(vmin=y.min(), vmax=y.max()))
    fig.colorbar(sm, ax=ax, label="output")
    plt.show()

## Function 1

In [ ]:
# ---- Function 1 ----
print_summary(1)
plot_marginals(1)
plot_2d_field(1)   # 2-D: full input-space colour map

## Function 2

In [ ]:
# ---- Function 2 ----
print_summary(2)
plot_marginals(2)
plot_2d_field(2)   # 2-D: full input-space colour map

## Function 3

In [ ]:
# ---- Function 3 ----
print_summary(3)
plot_marginals(3)
plot_pairs(3)       # pairwise scatter matrix (colour = output)
plot_parallel(3)    # parallel-coordinates view

## Function 4

In [ ]:
# ---- Function 4 ----
print_summary(4)
plot_marginals(4)
plot_pairs(4)       # pairwise scatter matrix (colour = output)
plot_parallel(4)    # parallel-coordinates view

## Function 5

In [ ]:
# ---- Function 5 ----
print_summary(5)
plot_marginals(5)
plot_pairs(5)       # pairwise scatter matrix (colour = output)
plot_parallel(5)    # parallel-coordinates view

## Function 6

In [ ]:
# ---- Function 6 ----
print_summary(6)
plot_marginals(6)
plot_pairs(6)       # pairwise scatter matrix (colour = output)
plot_parallel(6)    # parallel-coordinates view

## Function 7

In [ ]:
# ---- Function 7 ----
print_summary(7)
plot_marginals(7)
plot_pairs(7)       # pairwise scatter matrix (colour = output)
plot_parallel(7)    # parallel-coordinates view

## Function 8

In [ ]:
# ---- Function 8 ----
print_summary(8)
plot_marginals(8)
plot_pairs(8)       # pairwise scatter matrix (colour = output)
plot_parallel(8)    # parallel-coordinates view

## Quick strategy cheat-sheet

A starting map from the description hints to a BO setup. Treat these as
first guesses to validate against the plots above — not final answers.

| Fn | dim | Shape hint | Suggested acquisition | Notes |
|----|-----|------------|-----------------------|-------|
| 1 | 2 | sparse, needle-in-haystack, weak+strong sources | **UCB (high β)** / space-filling | outputs ≈ 0; explore first |
| 2 | 2 | noisy, multimodal | **noisy-EI** or UCB | fit a noise term (WhiteKernel) |
| 3 | 3 | costly, maximise −side-effects | EI | sample-efficient picks |
| 4 | 4 | multimodal, dynamic, local optima | UCB / EI (explore) | validate promising regions |
| 5 | 4 | **unimodal** (single peak) | **EI (small ξ)** / low-β UCB | easy; standardise the big-range target |
| 6 | 5 | additive negative penalties | EI + Matérn | smooth-ish surface |
| 7 | 6 | ML tuning, priors help | EI / UCB | log-warp scale-sensitive inputs |
| 8 | 8 | high-dim, target strong local maxima | **TuRBO / local BO** + UCB | global search is hard |

**On the project question — alternatives to Bayesian Optimisation for costly
functions:** the main families worth knowing are (a) **trust-region BO / TuRBO**
(BO variant that scales better in high dimensions — relevant for F7/F8),
(b) **CMA-ES** and other evolution strategies (derivative-free, robust on
multimodal surfaces, but hungrier for evaluations than BO), (c) **surrogate
optimisation with other models** — Random Forests (SMAC) or gradient-boosted
trees instead of a GP, which handle non-smooth/mixed inputs well, (d)
**DIRECT / Lipschitz-based** global optimisers, and (e) simple **Latin-hypercube
/ Sobol space-filling** for the pure-exploration phases. For truly expensive
evaluations and low dimensions (F1–F6), GP-based BO is still the most
sample-efficient default; the alternatives matter most as the dimension grows
(F7, F8) or when the surface is very rugged/noisy.

_Fill in the actual kernel/acquisition choices once you've eyeballed the plots
and started adding new sample points._